# Ecological Analysis Patterns

## Overview
This notebook applies spatial SQL and GeoPandas to real ecological analysis workflows: habitat overlap, species range estimation, observation density, and watershed-based aggregation — the kinds of queries that arise in conservation biology, environmental monitoring, and natural resource management.

**Analysis patterns covered:**
- Habitat overlap: intersecting protected areas with watershed boundaries
- Species range: convex hull of observations as a range proxy
- Protected vs unprotected observations: spatial join with protected area polygons
- Observation density per watershed (normalised by area)
- Upstream accumulation (PostGIS recursive CTE pattern)

---

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, Polygon, LineString, MultiPolygon
import pyproj

# ── Ecological dataset: Atlantic Canada ───────────────────────────
# Monitoring sites, species observations, protected areas, watersheds

# Monitoring sites (water quality / biodiversity)
sites_data = {
    'site_id':   ['S01','S02','S03','S04','S05','S06','S07','S08'],
    'site_name': ['Petitcodiac River','Miramichi Lake','Fundy Shore',
                  'Tantramar Marsh','Kouchibouguac','Restigouche River',
                  'Saint John River','Annapolis Valley'],
    'province':  ['NB','NB','NB','NB','NB','NB','NB','NS'],
    'elevation_m':[12, 84, 5, 3, 8, 45, 18, 62],
    'geometry':  [
        Point(-64.96, 46.07), Point(-66.18, 46.85), Point(-65.07, 45.55),
        Point(-64.40, 45.90), Point(-64.95, 46.82), Point(-67.60, 47.70),
        Point(-66.64, 45.95), Point(-65.12, 45.05),
    ],
}
sites = gpd.GeoDataFrame(sites_data, crs='EPSG:4326')

# Protected areas (simplified polygons)
prot_data = {
    'pa_id':    ['PA01','PA02','PA03'],
    'name':     ['Fundy National Park','Kouchibouguac NP','Kejimkujik NP'],
    'type':     ['National Park','National Park','National Park'],
    'area_km2': [207, 239, 404],
    'geometry': [
        Polygon([(-65.4,45.4),(-64.6,45.4),(-64.6,45.7),(-65.4,45.7)]),
        Polygon([(-64.9,46.6),(-64.7,46.6),(-64.7,47.0),(-64.9,47.0)]),
        Polygon([(-65.4,44.5),(-64.8,44.5),(-64.8,44.9),(-65.4,44.9)]),
    ],
}
protected = gpd.GeoDataFrame(prot_data, crs='EPSG:4326')

# Watersheds
ws_data = {
    'ws_id':   ['WS01','WS02','WS03'],
    'name':    ['Saint John Watershed','Miramichi Watershed','Petitcodiac Watershed'],
    'area_km2':[55200, 13800, 3100],
    'geometry':[
        Polygon([(-68.0,46.8),(-66.0,46.8),(-66.0,45.0),(-68.0,45.0)]),
        Polygon([(-66.5,47.5),(-65.5,47.5),(-65.5,46.5),(-66.5,46.5)]),
        Polygon([(-65.2,46.3),(-64.5,46.3),(-64.5,45.7),(-65.2,45.7)]),
    ],
}
watersheds = gpd.GeoDataFrame(ws_data, crs='EPSG:4326')

# Species observations
obs_data = {
    'obs_id':   range(1, 11),
    'species':  ['Atlantic Salmon','Atlantic Salmon','Blanding\'s Turtle',
                 'Piping Plover','Piping Plover','Blanding\'s Turtle',
                 'Atlantic Salmon','Little Brown Bat','Little Brown Bat','Piping Plover'],
    'year':     [2022,2023,2022,2023,2022,2023,2021,2023,2022,2021],
    'count':    [12, 8, 3, 1, 2, 1, 15, 47, 52, 3],
    'geometry': [
        Point(-64.96,46.07), Point(-67.60,47.70), Point(-64.40,45.90),
        Point(-65.07,45.55), Point(-64.95,46.82), Point(-66.18,46.85),
        Point(-66.64,45.95), Point(-66.18,46.85), Point(-64.96,46.07),
        Point(-64.40,45.90),
    ],
}
observations = gpd.GeoDataFrame(obs_data, crs='EPSG:4326')

print("Ecological dataset loaded:")
for name, gdf in [('sites',sites),('protected_areas',protected),
                  ('watersheds',watersheds),('observations',observations)]:
    print(f"  {name:<18s}: {len(gdf)} features, CRS={gdf.crs.to_epsg()}")


import geopandas as gpd
import pandas as pd

print("=== Habitat overlap analysis ===")
print()

# Compute overlap between each protected area and watershed
prot_utm = protected.to_crs("EPSG:32620")
ws_utm   = watersheds.to_crs("EPSG:32620")

print("Protected area / watershed overlap matrix:")
print(f"  {'Protected Area':<28s}  {'Watershed':<32s}  {'Overlap km²':>10s}  {'% of PA':>8s}")
print("  " + "-"*80)
for _, pa in prot_utm.iterrows():
    for _, ws in ws_utm.iterrows():
        overlap = pa.geometry.intersection(ws.geometry)
        if not overlap.is_empty and overlap.area > 0:
            overlap_km2 = overlap.area / 1e6
            pct_of_pa   = 100 * overlap.area / pa.geometry.area
            print(f"  {pa['name']:<28s}  {ws['name']:<32s}  {overlap_km2:>10.1f}  {pct_of_pa:>7.1f}%")

print()
print("PostGIS habitat overlap query:")
print("""
  SELECT pa.name        AS protected_area,
         ws.name        AS watershed,
         ST_Area(ST_Intersection(
             ST_Transform(pa.geom, 32620),
             ST_Transform(ws.geom, 32620)
         )) / 1e6       AS overlap_km2,
         100.0 * ST_Area(ST_Intersection(
             ST_Transform(pa.geom, 32620),
             ST_Transform(ws.geom, 32620)
         )) / ST_Area(ST_Transform(pa.geom, 32620))
                         AS pct_of_pa
  FROM   protected_areas pa
  JOIN   watersheds ws
      ON ST_Intersects(pa.geom, ws.geom)   -- use index; filter before ST_Area
  ORDER  BY overlap_km2 DESC;
""")


---
## Species range and observation patterns

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, Polygon, LineString, MultiPolygon
import pyproj

# ── Ecological dataset: Atlantic Canada ───────────────────────────
# Monitoring sites, species observations, protected areas, watersheds

# Monitoring sites (water quality / biodiversity)
sites_data = {
    'site_id':   ['S01','S02','S03','S04','S05','S06','S07','S08'],
    'site_name': ['Petitcodiac River','Miramichi Lake','Fundy Shore',
                  'Tantramar Marsh','Kouchibouguac','Restigouche River',
                  'Saint John River','Annapolis Valley'],
    'province':  ['NB','NB','NB','NB','NB','NB','NB','NS'],
    'elevation_m':[12, 84, 5, 3, 8, 45, 18, 62],
    'geometry':  [
        Point(-64.96, 46.07), Point(-66.18, 46.85), Point(-65.07, 45.55),
        Point(-64.40, 45.90), Point(-64.95, 46.82), Point(-67.60, 47.70),
        Point(-66.64, 45.95), Point(-65.12, 45.05),
    ],
}
sites = gpd.GeoDataFrame(sites_data, crs='EPSG:4326')

# Protected areas (simplified polygons)
prot_data = {
    'pa_id':    ['PA01','PA02','PA03'],
    'name':     ['Fundy National Park','Kouchibouguac NP','Kejimkujik NP'],
    'type':     ['National Park','National Park','National Park'],
    'area_km2': [207, 239, 404],
    'geometry': [
        Polygon([(-65.4,45.4),(-64.6,45.4),(-64.6,45.7),(-65.4,45.7)]),
        Polygon([(-64.9,46.6),(-64.7,46.6),(-64.7,47.0),(-64.9,47.0)]),
        Polygon([(-65.4,44.5),(-64.8,44.5),(-64.8,44.9),(-65.4,44.9)]),
    ],
}
protected = gpd.GeoDataFrame(prot_data, crs='EPSG:4326')

# Watersheds
ws_data = {
    'ws_id':   ['WS01','WS02','WS03'],
    'name':    ['Saint John Watershed','Miramichi Watershed','Petitcodiac Watershed'],
    'area_km2':[55200, 13800, 3100],
    'geometry':[
        Polygon([(-68.0,46.8),(-66.0,46.8),(-66.0,45.0),(-68.0,45.0)]),
        Polygon([(-66.5,47.5),(-65.5,47.5),(-65.5,46.5),(-66.5,46.5)]),
        Polygon([(-65.2,46.3),(-64.5,46.3),(-64.5,45.7),(-65.2,45.7)]),
    ],
}
watersheds = gpd.GeoDataFrame(ws_data, crs='EPSG:4326')

# Species observations
obs_data = {
    'obs_id':   range(1, 11),
    'species':  ['Atlantic Salmon','Atlantic Salmon','Blanding\'s Turtle',
                 'Piping Plover','Piping Plover','Blanding\'s Turtle',
                 'Atlantic Salmon','Little Brown Bat','Little Brown Bat','Piping Plover'],
    'year':     [2022,2023,2022,2023,2022,2023,2021,2023,2022,2021],
    'count':    [12, 8, 3, 1, 2, 1, 15, 47, 52, 3],
    'geometry': [
        Point(-64.96,46.07), Point(-67.60,47.70), Point(-64.40,45.90),
        Point(-65.07,45.55), Point(-64.95,46.82), Point(-66.18,46.85),
        Point(-66.64,45.95), Point(-66.18,46.85), Point(-64.96,46.07),
        Point(-64.40,45.90),
    ],
}
observations = gpd.GeoDataFrame(obs_data, crs='EPSG:4326')

print("Ecological dataset loaded:")
for name, gdf in [('sites',sites),('protected_areas',protected),
                  ('watersheds',watersheds),('observations',observations)]:
    print(f"  {name:<18s}: {len(gdf)} features, CRS={gdf.crs.to_epsg()}")


print("=== Species range and observation density ===")
print()

# Convex hull around each species' observations (range estimate)
obs_utm = observations.to_crs("EPSG:32620")
print("Species range (convex hull of observations):")
print(f"  {'species':<22s}  {'n_obs':>5s}  {'range_km²':>10s}  {'centroid (WGS84)'}")
print("  " + "-"*68)

for species, grp in obs_utm.groupby('species'):
    if len(grp) >= 3:
        hull = grp.geometry.union_all().convex_hull
        hull_wgs84 = gpd.GeoSeries([hull], crs='EPSG:32620').to_crs('EPSG:4326').iloc[0]
        area_km2 = hull.area / 1e6
        centroid = hull_wgs84.centroid
        print(f"  {species:<22s}  {len(grp):>5d}  {area_km2:>10.1f}  ({centroid.x:.2f}, {centroid.y:.2f})")
    else:
        print(f"  {species:<22s}  {len(grp):>5d}  {'< 3 obs':>10s}  (hull needs >= 3 points)")

print()
# Species within protected areas vs outside
print("Observations: inside vs outside protected areas:")
obs_wgs84 = observations.to_crs("EPSG:4326")
inside = gpd.sjoin(obs_wgs84, protected[['pa_id','name','geometry']], how='left', predicate='within')
inside['in_protected'] = inside['pa_id'].notna()

for species, grp in inside.groupby('species'):
    n_in  = grp['in_protected'].sum()
    n_out = (~grp['in_protected']).sum()
    print(f"  {species:<22s}  in PA: {n_in}  outside: {n_out}")

print()
print("PostGIS species range query:")
print("""
  -- Convex hull range per species:
  SELECT species,
         COUNT(*)                           AS n_observations,
         ST_Area(ST_Transform(
             ST_ConvexHull(ST_Collect(geom)), 32620
         )) / 1e6                           AS range_km2,
         ST_Centroid(ST_Collect(geom))      AS range_centroid
  FROM   species_observations
  GROUP  BY species
  ORDER  BY range_km2 DESC;

  -- Observations inside protected areas:
  SELECT o.species, pa.name AS protected_area, COUNT(*) AS n_obs
  FROM   species_observations o
  JOIN   protected_areas pa ON ST_Within(o.geom, pa.geom)
  GROUP  BY o.species, pa.name
  ORDER  BY n_obs DESC;
""")


---
## Watershed analysis

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, Polygon, LineString, MultiPolygon
import pyproj

# ── Ecological dataset: Atlantic Canada ───────────────────────────
# Monitoring sites, species observations, protected areas, watersheds

# Monitoring sites (water quality / biodiversity)
sites_data = {
    'site_id':   ['S01','S02','S03','S04','S05','S06','S07','S08'],
    'site_name': ['Petitcodiac River','Miramichi Lake','Fundy Shore',
                  'Tantramar Marsh','Kouchibouguac','Restigouche River',
                  'Saint John River','Annapolis Valley'],
    'province':  ['NB','NB','NB','NB','NB','NB','NB','NS'],
    'elevation_m':[12, 84, 5, 3, 8, 45, 18, 62],
    'geometry':  [
        Point(-64.96, 46.07), Point(-66.18, 46.85), Point(-65.07, 45.55),
        Point(-64.40, 45.90), Point(-64.95, 46.82), Point(-67.60, 47.70),
        Point(-66.64, 45.95), Point(-65.12, 45.05),
    ],
}
sites = gpd.GeoDataFrame(sites_data, crs='EPSG:4326')

# Protected areas (simplified polygons)
prot_data = {
    'pa_id':    ['PA01','PA02','PA03'],
    'name':     ['Fundy National Park','Kouchibouguac NP','Kejimkujik NP'],
    'type':     ['National Park','National Park','National Park'],
    'area_km2': [207, 239, 404],
    'geometry': [
        Polygon([(-65.4,45.4),(-64.6,45.4),(-64.6,45.7),(-65.4,45.7)]),
        Polygon([(-64.9,46.6),(-64.7,46.6),(-64.7,47.0),(-64.9,47.0)]),
        Polygon([(-65.4,44.5),(-64.8,44.5),(-64.8,44.9),(-65.4,44.9)]),
    ],
}
protected = gpd.GeoDataFrame(prot_data, crs='EPSG:4326')

# Watersheds
ws_data = {
    'ws_id':   ['WS01','WS02','WS03'],
    'name':    ['Saint John Watershed','Miramichi Watershed','Petitcodiac Watershed'],
    'area_km2':[55200, 13800, 3100],
    'geometry':[
        Polygon([(-68.0,46.8),(-66.0,46.8),(-66.0,45.0),(-68.0,45.0)]),
        Polygon([(-66.5,47.5),(-65.5,47.5),(-65.5,46.5),(-66.5,46.5)]),
        Polygon([(-65.2,46.3),(-64.5,46.3),(-64.5,45.7),(-65.2,45.7)]),
    ],
}
watersheds = gpd.GeoDataFrame(ws_data, crs='EPSG:4326')

# Species observations
obs_data = {
    'obs_id':   range(1, 11),
    'species':  ['Atlantic Salmon','Atlantic Salmon','Blanding\'s Turtle',
                 'Piping Plover','Piping Plover','Blanding\'s Turtle',
                 'Atlantic Salmon','Little Brown Bat','Little Brown Bat','Piping Plover'],
    'year':     [2022,2023,2022,2023,2022,2023,2021,2023,2022,2021],
    'count':    [12, 8, 3, 1, 2, 1, 15, 47, 52, 3],
    'geometry': [
        Point(-64.96,46.07), Point(-67.60,47.70), Point(-64.40,45.90),
        Point(-65.07,45.55), Point(-64.95,46.82), Point(-66.18,46.85),
        Point(-66.64,45.95), Point(-66.18,46.85), Point(-64.96,46.07),
        Point(-64.40,45.90),
    ],
}
observations = gpd.GeoDataFrame(obs_data, crs='EPSG:4326')

print("Ecological dataset loaded:")
for name, gdf in [('sites',sites),('protected_areas',protected),
                  ('watersheds',watersheds),('observations',observations)]:
    print(f"  {name:<18s}: {len(gdf)} features, CRS={gdf.crs.to_epsg()}")


print("=== Watershed analysis patterns ===")
print()

ws_utm   = watersheds.to_crs("EPSG:32620")
sites_utm = sites.to_crs("EPSG:32620")
obs_utm   = observations.to_crs("EPSG:32620")

# Sites per watershed with elevation stats
sites_in_ws = gpd.sjoin(
    sites_utm[['site_id','site_name','elevation_m','geometry']],
    ws_utm[['ws_id','name','area_km2','geometry']],
    how='left', predicate='within'
)
print("Monitoring sites per watershed with elevation stats:")
summary = (sites_in_ws.dropna(subset=['ws_id'])
           .groupby(['ws_id','name','area_km2'])
           .agg(n_sites=('site_id','count'),
                avg_elev=('elevation_m','mean'),
                max_elev=('elevation_m','max'))
           .reset_index())
for _, row in summary.iterrows():
    print(f"  {row.ws_id} {row['name']:<32s}  "
          f"sites={row.n_sites}  avg_elev={row.avg_elev:.0f}m  max_elev={row.max_elev}m")

print()
# Observation density per km² per watershed
obs_in_ws = gpd.sjoin(
    obs_utm[['obs_id','species','count','geometry']],
    ws_utm[['ws_id','name','area_km2','geometry']],
    how='left', predicate='within'
)
print("Species observation density per watershed:")
density = (obs_in_ws.dropna(subset=['ws_id'])
           .groupby(['ws_id','name','area_km2'])
           .agg(n_obs=('obs_id','count'), total_count=('count','sum'))
           .reset_index())
density['obs_per_1000km2'] = 1000 * density['n_obs'] / density['area_km2'].astype(float)
for _, row in density.iterrows():
    print(f"  {row['name']:<32s}  "
          f"obs={row.n_obs}  total_count={row.total_count}  "
          f"density={row.obs_per_1000km2:.2f}/1000km²")

print()
print("PostGIS watershed analysis patterns:")
print("""
  -- Site summary per watershed:
  SELECT w.name, w.area_km2,
         COUNT(s.site_id) AS n_sites,
         AVG(s.elevation_m) AS avg_elevation
  FROM   watersheds w
  LEFT JOIN monitoring_sites s ON ST_Within(s.geom, w.geom)
  GROUP  BY w.ws_id, w.name, w.area_km2;

  -- Upstream accumulation (requires stream network + direction):
  WITH RECURSIVE upstream AS (
      SELECT stream_id, geom, 0 AS level
      FROM   streams WHERE stream_id = 'OUTLET_001'
      UNION ALL
      SELECT s.stream_id, s.geom, u.level + 1
      FROM   streams s
      JOIN   upstream u ON ST_Touches(s.geom, u.geom) AND s.flow_direction = 'into'
  )
  SELECT COUNT(*), SUM(length_km) FROM upstream;
""")


---
## Common Pitfalls

**1. Convex hull overestimating species range**
A convex hull includes all land between observation points, even unsuitable habitat. A species observed at two coastal points will have a hull extending far inland. Use kernel density estimation or alpha shapes for more realistic range models.

**2. Not accounting for observer effort in density maps**
High observation density near roads or research stations reflects sampling effort, not true species abundance. Normalise by effort (survey hours, transect length) before interpreting observation density maps.

**3. Spatial join at wrong scale for watershed analysis**
Using national watershed boundaries (coarse polygons) for site-level analysis misassigns sites near watershed boundaries. Use the finest-scale watershed layer available for the analysis region.

**4. Ignoring the modifiable areal unit problem (MAUP)**
Aggregating point observations to grid cells or administrative areas changes results depending on the boundary choice. The same observation data can show different patterns depending on whether you aggregate to 1km, 10km, or watershed polygons.

**5. Treating date as independent when computing species range over time**
Combining observations from different years into one range polygon obscures range shifts. Always include year/date as a dimension and compute annual or seasonal ranges separately before comparing across time.

**6. Missing coordinate validation for field-collected observations**
Field GPS coordinates often contain errors: `(0, 0)` (null island), coordinates outside the study region, or swapped lat/lon. Always validate: `WHERE ST_Within(geom, ST_MakeEnvelope(study_bbox, 4326))` and `WHERE geom IS NOT NULL` before including in analysis.


---
*sql_methods_library - Samantha McGarrigle*